# Git Revert: Use Cases and Visual Examples

This notebook explains how `git revert` works, when to use it, and provides visual examples for different scenarios involving local and remote branches.



## What is `git revert`?

- `git revert` creates a new commit that undoes the changes made by a previous commit.
- It **does not** delete commits or rewrite history; instead, it adds a new commit that "reverses" the effect of the specified commit(s).
- This is a **safe** way to undo changes, especially on branches that are shared with others.



## Common Use Cases for `git revert`

### 1. Undo the Latest Commit
```bash
git revert HEAD
```
- Reverts the latest commit on your current branch.
- Safe for shared branches.

### 2. Undo a Specific Commit
```bash
git revert <commit-hash>
```
- Reverts a specific commit in the past.

### 3. Undo Multiple Commits (a range)
```bash
git revert <oldest-commit-hash>^..<newest-commit-hash>
```
- Reverts a series of consecutive commits.

### 4. Undo a Merge Commit
```bash
git revert -m 1 <merge-commit-hash>
```
- Reverts a merge commit. The `-m` flag specifies the parent branch to keep (usually `1` for the main branch).

### 5. Undo a Commit on a Shared Branch
- Use `git revert` instead of `git reset` to avoid rewriting history on branches others may use.



## When **Not** to Use `git revert`

- If you want to **remove commits entirely** (as if they never happened), use `git reset` (but only on local/unshared branches).
- If you want to rewrite history (e.g., squash, reorder, or delete commits), use `git rebase` or `git reset` (again, only on local/unshared branches).



## Visual Examples: Local and Remote Branch Status

Below are visual diagrams showing the state of your local and remote branches **before and after** running `git revert HEAD` in different scenarios.



### 1. No local commits, up to date with remote

**Before:**
```
Remote:   A -- B -- C  (origin/main)
Local:    A -- B -- C  (main, HEAD)
```

**After `git revert HEAD`:**
```
Remote:   A -- B -- C  (origin/main)
Local:    A -- B -- C -- D  (main, HEAD)
```
- D is a new commit that undoes the changes from C.
- You need to push D to update the remote.
- **After revert, your real code looks like it did after B (but with a new commit D in history).**



### 2. Local commits not pushed

**Before:**
```
Remote:   A -- B -- C  (origin/main)
Local:    A -- B -- C -- D -- E  (main, HEAD)
```
- D and E are local commits, not pushed.
- `HEAD` points to E.

**After `git revert HEAD`:**
```
Remote:   A -- B -- C  (origin/main)
Local:    A -- B -- C -- D -- E -- F  (main, HEAD)
```
- F is a new commit that undoes the changes from E.
- D and E are still in your history, but E’s changes are "undone" by F.
- You need to push D, E, and F to update the remote.
- **After revert, your real code looks like it did after D (but with E and F in history).**



### 3. Local and remote are the same

**Before:**
```
Remote:   A -- B -- C  (origin/main)
Local:    A -- B -- C  (main, HEAD)
```

**After `git revert HEAD`:**
```
Remote:   A -- B -- C  (origin/main)
Local:    A -- B -- C -- D  (main, HEAD)
```
- D is a new commit that undoes C.
- You need to push D to update the remote.
- **After revert, your real code looks like it did after B (but with a new commit D in history).**



## What happens after revert?

- The commit you reverted is **not deleted**; it’s still in the history.
- A new commit is created that undoes the changes of the reverted commit.
- Your branch moves forward by one commit.
- The remote branch is **not changed** until you push.



## Summary Table

| Situation                  | Before (Local)         | Before (Remote)      | After `git revert HEAD` (Local) | After (Remote, before push) |
|----------------------------|------------------------|----------------------|-------------------------------|-----------------------------|
| Up to date                 | A--B--C (HEAD)         | A--B--C              | A--B--C--D (HEAD)             | A--B--C                     |
| Local commits not pushed    | A--B--C--D--E (HEAD)   | A--B--C              | A--B--C--D--E--F (HEAD)       | A--B--C                     |
| Local and remote are same   | A--B--C (HEAD)         | A--B--C              | A--B--C--D (HEAD)             | A--B--C                     |

- `git revert HEAD` always reverts your current local latest commit, whether it’s pushed or not.
- Use `git revert` for safe, auditable undoing of changes, especially on shared branches.



## Advanced Use Cases for `git revert`

Below are some more complex scenarios, including reverting by commit ID, reverting merge commits, and using `HEAD~n` syntax.



### 1. Reverting a Specific Commit by Commit ID

Suppose your history is:
```
A -- B -- C -- D -- E (HEAD)
```
You want to revert commit C (not the latest).

**Command:**
```bash
git revert <commit-hash-of-C>
```

**Result:**
```
A -- B -- C -- D -- E -- F (HEAD)
```
- F is a new commit that undoes the changes from C.
- **After revert, your real code looks like it did after B, D, and E (as if C never happened, but C and F are both in history).**
- The changes from D and E remain.



### 2. Reverting a Merge Commit

Suppose your history is:
```
A -- B -- C -- M (HEAD)
           \   /
            F--G
```
- M is a merge commit that merged feature branch (F, G) into main.

**Command:**
```bash
git revert -m 1 <merge-commit-hash-of-M>
```
- The `-m 1` option tells Git to keep the main branch's changes (parent 1) and revert the merge.

**Result:**
```
A -- B -- C -- M -- R (HEAD)
           \   /
            F--G
```
- R is a new commit that undoes the changes introduced by the merge commit M.
- **After revert, your real code looks like it did after C (as if the merge never happened, but M and R are both in history).**



### 3. Reverting Using `HEAD~n` Syntax

Suppose your history is:
```
A -- B -- C -- D -- E (HEAD)
```
- `HEAD~1` refers to D (one commit before HEAD)
- `HEAD~2` refers to C (two commits before HEAD)

**Command:**
```bash
git revert HEAD~2
```
- This reverts commit C.

**Result:**
```
A -- B -- C -- D -- E -- F (HEAD)
```
- F is a new commit that undoes the changes from C.
- **After revert, your real code looks like it did after B, D, and E (as if C never happened, but C and F are both in history).**



### 4. Reverting a Range of Commits

Suppose your history is:
```
A -- B -- C -- D -- E (HEAD)
```
You want to revert both C and D.

**Command:**
```bash
git revert C^..D
```
- This will create two new revert commits (one for C, one for D).

**Result:**
```
A -- B -- C -- D -- E -- F -- G (HEAD)
```
- F undoes C, G undoes D.
- **After revert, your real code looks like it did after B and E (as if C and D never happened, but all commits are in history).**



### 5. Reverting a Range of the Last N Commits

Suppose your history is:
```
A -- B -- C -- D -- E -- F (HEAD)
```

If you want to revert the **last 5 commits** (B, C, D, E, F):

**Command:**
```bash
git revert HEAD~4..HEAD
```
- This will create a new revert commit for each of the last 5 commits.

**Result:**
```
A -- B -- C -- D -- E -- F -- G -- H -- I -- J -- K (HEAD)
```
- G, H, I, J, K are revert commits for B, C, D, E, F respectively.
- **After revert, your real code looks like it did after A (but all original and revert commits are in history).**

**Note:**
- `git revert HEAD~2` only reverts the single commit at `HEAD~2` (the third most recent), not a range.
- To revert a range, always use the `..` syntax as shown above.



## Useful `git revert` Options and Examples

Below are some important options you can use with `git revert`, along with examples for each.



### 1. `--no-edit`

- **What it does:** Uses the default commit message for the revert, without opening an editor.
- **When to use:** For scripting, automation, or when you don’t want to edit the message.

**Example:**
```bash
git revert HEAD --no-edit
```
- This reverts the latest commit and uses the default message automatically.



### 2. `--edit` (default)

- **What it does:** Opens your editor to let you modify the revert commit message.
- **When to use:** If you want to add extra context or explanation to the revert commit.

**Example:**
```bash
git revert HEAD --edit
```
- This is the default behavior if you don’t specify `--no-edit`.



### 3. `--no-commit` (or `-n`)

- **What it does:** Applies the revert changes to your working directory and staging area, but does **not** create a commit.
- **When to use:** If you want to review, modify, or combine multiple reverts into a single commit.

**Example:**
```bash
git revert HEAD~2..HEAD --no-commit
git commit -m "Revert last three commits as a single commit"
```
- This reverts the last three commits, stages the changes, and lets you commit them all together.



### 4. `-m <parent-number>` (for merge commits)

- **What it does:** Required when reverting a merge commit. Specifies which parent branch to keep.
- **When to use:** When you need to revert a merge commit.

**Example:**
```bash
git revert -m 1 <merge-commit-hash>
```
- This reverts the merge commit, keeping the changes from parent 1 (usually the main branch).



### 5. `--continue`, `--abort`, `--quit` (for resolving conflicts)

- **What they do:** Used if a revert causes conflicts.
    - `--continue`: After resolving conflicts, continue the revert process.
    - `--abort`: Cancel the revert and return to the previous state.
    - `--quit`: Stop the revert process but keep the index and working tree as they are.

**Example (conflict resolution):**
```bash
git revert HEAD~2..HEAD
# If there are conflicts:
# (Resolve conflicts in your editor)
git revert --continue
# Or, to abort:
git revert --abort
```

